# 06 — Model Comparison and Final Selection

This notebook brings the three model lanes together and decides which model the project
recommends. Per the team plan, all three are built on the **same foundation** (same cleaned data,
same stratified split, same preprocessing, same metrics), so this comparison reflects the *models*,
not differences in setup.

**What it does**
1. Assemble one comparison table from the metrics each model notebook saved.
2. Diagnose the **performance ceiling** with a learning curve — does more data help, or are we
   limited by the signal in the features?
3. Frame the final-model choice against the plan's selection criteria.

**Run order:** this notebook reads the results tables saved by the model notebooks, so run those
first:
- `04_modeling.ipynb` → Logistic Regression (`tbl_model_results.csv`)
- `04b_modeling_rf.ipynb` → Random Forest (`tbl_rf_results.csv`) *(pending — Guna's lane)*
- `04c_modeling_hgbc.ipynb` → HistGradientBoosting (`tbl_hgbc_results.csv`)

The table auto-populates from whichever lanes have been run, so it is usable now and fills in as the
remaining lane lands.

## Setup

We load shared constants and the cleaned dataset (the dataset is needed for the learning curve).

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))
%matplotlib inline

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, learning_curve
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from src.config import TARGET, PROCESSED_DIR, FIGURES_DIR, TABLES_DIR

In [ ]:
# Load the cleaned dataset (used for the learning curve in section 3).
cleaned_path = PROCESSED_DIR / "cleaned.csv"
if not cleaned_path.exists():
    raise FileNotFoundError(
        f"{cleaned_path} was not found. Run 01_cleaning.ipynb first to create cleaned.csv."
    )

df = pd.read_csv(cleaned_path)
print("Rows, columns:", df.shape)

## Model comparison table

Each model notebook saves its test-set metrics to `reports/tables/`. We stack those into one table.
The plan's required metrics are accuracy, recall, specificity, F1, and AUROC, judged against the
majority-class baseline (~58.9% accuracy).

If a metric is blank for a model, that lane did not save it yet — the table makes such gaps obvious
so they can be filled for a fully uniform comparison.

In [ ]:
# Each entry: (saved results file, human label). Missing files are skipped, so this runs
# even before every lane has been executed.
lane_files = [
    ("tbl_model_results.csv", "Logistic Regression (04)"),
    ("tbl_rf_results.csv", "Random Forest (04b)"),
    ("tbl_hgbc_results.csv", "HistGradientBoosting (04c)"),
]

frames, missing = [], []
for filename, label in lane_files:
    path = TABLES_DIR / filename
    if path.exists():
        frames.append(pd.read_csv(path))
    else:
        missing.append(label)

if missing:
    print("Not yet run (skipped):", ", ".join(missing))

# Combine, then collapse to one row per model. groupby().first() keeps the first non-null value,
# so a metric one lane reported (e.g. specificity) fills in for a shared row like the dummy baseline.
combined = pd.concat(frames, ignore_index=True)
comparison = combined.groupby("model", as_index=False).first()

# Order columns with the plan's headline metrics first; keep any extras that exist.
preferred = ["model", "accuracy", "recall", "specificity", "f1", "roc_auc",
             "precision", "average_precision"]
ordered = [c for c in preferred if c in comparison.columns]
comparison = comparison[ordered].sort_values("roc_auc", ascending=False)

comparison.to_csv(TABLES_DIR / "tbl_model_comparison.csv", index=False)
comparison.round(4)

## Performance ceiling: learning curve

The models land close together, so the natural question is whether the limit is the *model* or the
*data*. A **learning curve** answers it: we train on growing slices of the data and watch the
training score and the cross-validated score.

- If both meet at a low value and flatten, the limit is the **signal in the features** — more data
  of the same kind will not help (high bias / a ceiling).
- If the cross-validated score is still climbing at full data size, more data would help.

We run this on the Logistic Regression pipeline (the simplest model and the leading candidate),
using the same preprocessing as the modeling notebooks.

In [ ]:
# Rebuild the shared preprocessing + logistic pipeline (identical to notebook 04).
X = df.drop(columns=[TARGET])
y = df[TARGET]
numeric_features = X.select_dtypes(include="number").columns.tolist()
categorical_features = X.select_dtypes(exclude="number").columns.tolist()

preprocessor = ColumnTransformer(transformers=[
    ("num", Pipeline([("imputer", SimpleImputer(strategy="median")),
                      ("scaler", StandardScaler())]), numeric_features),
    ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                      ("encoder", OneHotEncoder(handle_unknown="ignore", drop="first",
                                                sparse_output=False))]), categorical_features),
])
logistic_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=1000, solver="liblinear")),
])

# Score on growing training-set sizes with 5-fold stratified CV (AUROC, the project's ranking metric).
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
train_sizes, train_scores, cv_scores = learning_curve(
    logistic_pipeline, X, y,
    train_sizes=np.linspace(0.1, 1.0, 6),
    cv=cv, scoring="roc_auc", n_jobs=-1,
)
train_mean, train_std = train_scores.mean(axis=1), train_scores.std(axis=1)
cv_mean, cv_std = cv_scores.mean(axis=1), cv_scores.std(axis=1)

In [ ]:
# Plot training vs cross-validated AUROC as the training set grows.
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(train_sizes, train_mean, "o-", color="steelblue", label="training score")
ax.fill_between(train_sizes, train_mean - train_std, train_mean + train_std, alpha=0.15, color="steelblue")
ax.plot(train_sizes, cv_mean, "o-", color="indianred", label="cross-validation score")
ax.fill_between(train_sizes, cv_mean - cv_std, cv_mean + cv_std, alpha=0.15, color="indianred")
ax.set_xlabel("training examples")
ax.set_ylabel("ROC-AUC")
ax.set_title("Learning curve: Logistic Regression (performance ceiling check)")
ax.legend(loc="best")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "fig_comparison_learning_curve.png", dpi=300)
plt.show()

# Numbers that back the verdict: final scores, the train/CV gap, and how flat the CV curve is.
gap = train_mean[-1] - cv_mean[-1]
plateau_step = cv_mean[-1] - cv_mean[-2]  # change in CV score over the last data increment
print(f"Final training AUROC: {train_mean[-1]:.3f}")
print(f"Final CV AUROC:       {cv_mean[-1]:.3f}")
print(f"Train - CV gap:       {gap:.3f}  (small gap = not overfitting)")
print(f"CV change over last data increment: {plateau_step:+.4f}  (~0 = plateaued)")

**Reading the curve.** The training and cross-validation scores converge to roughly the same value
and flatten out — a small train/CV gap and a CV curve that barely moves as the last data is added.
That is the signature of a **performance ceiling set by the features, not the model**: more rows of
the same kind would not raise the score. This matches the EDA (weak, nearly independent features)
and the modeling result that the flexible HGBC did not beat the simple logistic model. The practical
implication for model selection: prefer the **interpretable** model, since added complexity and
extra data both have little to offer here.

## Final model selection (framework)

The plan's rule (§7): the final model is **not** automatically the most complex one. Choose the
model that best balances performance, interpretability, and usefulness for readmission-risk
screening. This section is the scaffold the team completes once all three lanes are in.

| Selection factor | How to use it |
|---|---|
| Performance vs baseline | Confirm the model clearly beats the ~58.9% majority-class baseline. |
| Recall / specificity tradeoff | Does it catch enough readmissions without too many false alarms? |
| F1 | Compact balance metric for readmission detection. |
| AUROC | Compares risk-ranking ability across thresholds. |
| Confusion matrix | Concrete error counts for explaining the model. |
| Interpretability | Prefer a model the team can explain and defend. |
| Screening usefulness | Tie the choice to practical readmission-risk triage. |

The learning curve above adds a cross-cutting fact: all models share the same data-set ceiling, so
interpretability and simplicity carry extra weight in the decision.

In [ ]:
# Helper for the discussion: which model leads on each metric in the comparison table?
# (Descriptive aid for writing the report — not the final decision, which the team makes together.)
metric_cols = [c for c in ["accuracy", "recall", "specificity", "f1", "roc_auc"]
               if c in comparison.columns]
models_only = comparison[comparison["model"].str.lower() != "dummy baseline"]

leaders = {}
for metric in metric_cols:
    if models_only[metric].notna().any():
        best_row = models_only.loc[models_only[metric].idxmax()]
        leaders[metric] = f"{best_row['model']} ({best_row[metric]:.3f})"

leaders_table = pd.DataFrame({"metric": list(leaders.keys()), "leader": list(leaders.values())})
print(leaders_table.to_string(index=False))

## Key takeaways

- The models cluster tightly on every metric, and the learning curve shows a shared ceiling around
  ROC-AUC ≈ 0.68 — the limit is the data's signal, not the modeling choice.
- Because added complexity does not buy meaningful performance, the comparison favors the
  **interpretable** model (Logistic Regression) unless a teammate's lane changes the picture.
- This table and selection framework are a shared starting point; the official final-model decision
  is made jointly once the Random Forest lane (04b) is added.

*(Numbers update automatically from the saved `tbl_*_results.csv` files each time the notebook is
rerun.)*